# Optunaによるパラメータのオートチューニング

In [1]:
!pip install -qq optuna kaggle-environments

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.7/721.7 kB 7.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.4/442.4 kB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 944.3/944.3 kB 42.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.3/96.3 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 65.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 840.2/840.2 kB 36.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.4/111.4 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.4/267.4 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.2/20.2 MB 63.1 MB

In [2]:
"""Colabで実行するパラメータ探索コード。"""

import importlib
import statistics

import optuna
from kaggle_environments import make

import main as strategy


# Colab上の最新のmain.pyを読み込む。
strategy = importlib.reload(strategy)

N_TRIALS = 150
MATCH_COUNT = 20
EVALUATION_SEEDS = list(range(MATCH_COUNT))


def objective(trial):
    """平均得点を返す。"""

    #メロンを収穫対象として扱い始める経過日数
    strategy.StrategyConfig.MELON_HARVEST_AGE = trial.suggest_int(
        "MELON_HARVEST_AGE",
        10,
        12,
    )

    #いちごを収穫対象として扱い始める経過日数
    strategy.StrategyConfig.STRAWBERRY_HARVEST_AGE = trial.suggest_int(
        "STRAWBERRY_HARVEST_AGE",
        10,
        14,
        step=2,
    )

    #人参を収穫対象として扱い始める経過日数
    strategy.StrategyConfig.CARROT_HARVEST_AGE = trial.suggest_int(
        "CARROT_HARVEST_AGE",
        2,
        3,
    )

    #メロン・いちご以外の作物に使用する収穫経過日数
    strategy.StrategyConfig.DEFAULT_HARVEST_AGE = trial.suggest_int(
        "DEFAULT_HARVEST_AGE",
        2,
        4,
    )


    rewards = []

    for episode_seed in EVALUATION_SEEDS:
        strategy.hire_controller = strategy.HireController()

        env = make(
            "kaggriculture",
            configuration={
                "episodeSteps": 720,
                "seed": episode_seed,
            },
            debug=True,
        )

        env.run([strategy.agent, strategy.agent])

        for state in env.steps[-1]:
            rewards.append(float(state.reward))

    return statistics.fmean(rewards)


study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
)

study.optimize(
    objective,
    n_trials=N_TRIALS,
    n_jobs=1,
    show_progress_bar=True,
)

print("\n===== 最高平均得点 =====")
print(study.best_value)

print("\n===== main.pyへ手動設定する値 =====")

for name, value in study.best_params.items():
    print(f"{name} = {value}")

[I 2026-09-10 09:40:57,916] A new study created in memory with name: no-name-345956cf-1c8e-43c4-9217-cd9761f0b3ee


  0%|          | 0/150 [00:00<?, ?it/s]

[I 2026-09-10 09:44:16,462] Trial 0 finished with value: 55642.0 and parameters: {'MELON_HARVEST_AGE': 11, 'STRAWBERRY_HARVEST_AGE': 14, 'CARROT_HARVEST_AGE': 3, 'DEFAULT_HARVEST_AGE': 3}. Best is trial 0 with value: 55642.0.
[I 2026-09-10 09:47:35,674] Trial 1 finished with value: 61615.475 and parameters: {'MELON_HARVEST_AGE': 10, 'STRAWBERRY_HARVEST_AGE': 10, 'CARROT_HARVEST_AGE': 2, 'DEFAULT_HARVEST_AGE': 4}. Best is trial 1 with value: 61615.475.
[I 2026-09-10 09:50:50,246] Trial 2 finished with value: 54599.3 and parameters: {'MELON_HARVEST_AGE': 11, 'STRAWBERRY_HARVEST_AGE': 14, 'CARROT_HARVEST_AGE': 2, 'DEFAULT_HARVEST_AGE': 4}. Best is trial 1 with value: 61615.475.
[I 2026-09-10 09:54:03,275] Trial 3 finished with value: 60880.7 and parameters: {'MELON_HARVEST_AGE': 12, 'STRAWBERRY_HARVEST_AGE': 10, 'CARROT_HARVEST_AGE': 2, 'DEFAULT_HARVEST_AGE': 2}. Best is trial 1 with value: 61615.475.
[I 2026-09-10 09:57:15,666] Trial 4 finished with value: 67529.5 and parameters: {'MELON